In [1]:
!pip install torch torchvision kornia opencv-python matplotlib

   ---------------------------------------- 0.0/1.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.1 MB ? eta -:--:--
   -------------------------------

In [1]:
import os
import gc
from pathlib import Path
from typing import List
import cv2
import numpy as np
import matplotlib.pyplot as plt
import torch
import kornia as K
import kornia.feature as KF

In [ ]:
# 1. CONFIGURATION

IMAGE_FOLDER = "eg_images"   # change to your folder
IMAGE_EXTENSIONS = [".jpg", ".jpeg", ".png"]

OUTPUT_FOLDER = "deep_stitching_output"
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

# smaller size to avoid kernel crash
MAX_WIDTH = 640
MAX_HEIGHT = 480

# safer than auto cuda
DEVICE = torch.device("cpu")


# 2. HELPERS

def resize_keep_ratio(img, max_width=640, max_height=480):
    h, w = img.shape[:2]
    scale = min(max_width / w, max_height / h, 1.0)
    new_w = int(w * scale)
    new_h = int(h * scale)
    return cv2.resize(img, (new_w, new_h))

def load_images(folder: str, max_images: int = 4) -> List[np.ndarray]:
    if not os.path.exists(folder):
        raise FileNotFoundError(f"Folder not found: {folder}")

    images = []
    files = sorted(os.listdir(folder))

    for file in files[:max_images]:
        full_path = os.path.join(folder, file)
        if os.path.isfile(full_path) and Path(file).suffix.lower() in IMAGE_EXTENSIONS:
            img = cv2.imread(full_path)
            if img is not None:
                img = resize_keep_ratio(img, MAX_WIDTH, MAX_HEIGHT)
                images.append(img)
                print(f"Loaded: {file} | shape: {img.shape}")
            else:
                print(f"Could not read: {file}")

    print(f"Total images loaded: {len(images)}")
    return images

def show_images(images: List[np.ndarray], titles=None, cols=2, figsize=(12, 8)):
    if len(images) == 0:
        return
    rows = int(np.ceil(len(images) / cols))
    plt.figure(figsize=figsize)
    for i, img in enumerate(images):
        plt.subplot(rows, cols, i + 1)
        plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        plt.title(titles[i] if titles else f"Image {i+1}")
        plt.axis("off")
    plt.tight_layout()
    plt.show()

def save_image(img: np.ndarray, filename: str):
    save_path = os.path.join(OUTPUT_FOLDER, filename)
    cv2.imwrite(save_path, img)
    print(f"Saved: {save_path}")

def crop_black_borders(img: np.ndarray) -> np.ndarray:
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    _, thresh = cv2.threshold(gray, 1, 255, cv2.THRESH_BINARY)
    coords = cv2.findNonZero(thresh)
    if coords is None:
        return img
    x, y, w, h = cv2.boundingRect(coords)
    return img[y:y+h, x:x+w]

def numpy_to_torch_gray(img: np.ndarray) -> torch.Tensor:
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    tensor = K.image_to_tensor(img_rgb, keepdim=False).float() / 255.0
    tensor = tensor.to(DEVICE)
    gray = K.color.rgb_to_grayscale(tensor)
    return gray


# 3. LOAD LoFTR

matcher = KF.LoFTR(pretrained="outdoor").to(DEVICE).eval()

def deep_match_loftr(img1: np.ndarray, img2: np.ndarray):
    t1 = numpy_to_torch_gray(img1)
    t2 = numpy_to_torch_gray(img2)

    input_dict = {
        "image0": t1,
        "image1": t2,
    }

    with torch.no_grad():
        correspondences = matcher(input_dict)

    mkpts0 = correspondences["keypoints0"].cpu().numpy()
    mkpts1 = correspondences["keypoints1"].cpu().numpy()

    del t1, t2, input_dict, correspondences
    gc.collect()

    return mkpts0, mkpts1


# 4. HOMOGRAPHY + STITCH

def compute_homography_from_matches(pts_src, pts_dst, ransac_thresh=4.0):
    if len(pts_src) < 4 or len(pts_dst) < 4:
        return None, None

    H, mask = cv2.findHomography(
        pts_dst.reshape(-1, 1, 2),
        pts_src.reshape(-1, 1, 2),
        cv2.RANSAC,
        ransac_thresh
    )
    return H, mask

def stitch_two_images(img1: np.ndarray, img2: np.ndarray, H: np.ndarray):
    h1, w1 = img1.shape[:2]
    h2, w2 = img2.shape[:2]

    corners_img2 = np.float32([[0, 0], [0, h2], [w2, h2], [w2, 0]]).reshape(-1, 1, 2)
    warped_corners_img2 = cv2.perspectiveTransform(corners_img2, H)

    corners_img1 = np.float32([[0, 0], [0, h1], [w1, h1], [w1, 0]]).reshape(-1, 1, 2)
    all_corners = np.concatenate((corners_img1, warped_corners_img2), axis=0)

    [xmin, ymin] = np.int32(all_corners.min(axis=0).ravel() - 0.5)
    [xmax, ymax] = np.int32(all_corners.max(axis=0).ravel() + 0.5)

    translation = np.array([
        [1, 0, -xmin],
        [0, 1, -ymin],
        [0, 0, 1]
    ], dtype=np.float32)

    out_w = xmax - xmin
    out_h = ymax - ymin

    warped_img2 = cv2.warpPerspective(img2, translation @ H, (out_w, out_h))
    result = warped_img2.copy()

    x_offset = -xmin
    y_offset = -ymin

    roi = result[y_offset:y_offset+h1, x_offset:x_offset+w1]

    img1_mask = np.any(img1 > 0, axis=2)
    roi_mask = np.any(roi > 0, axis=2)

    overlap = img1_mask & roi_mask
    only_img1 = img1_mask & ~roi_mask

    blended_roi = roi.copy()
    blended_roi[only_img1] = img1[only_img1]
    blended_roi[overlap] = (
        0.5 * roi[overlap].astype(np.float32) +
        0.5 * img1[overlap].astype(np.float32)
    ).astype(np.uint8)

    result[y_offset:y_offset+h1, x_offset:x_offset+w1] = blended_roi
    return result

def deep_stitch_all(images: List[np.ndarray]):
    panorama = images[0].copy()

    for i in range(1, len(images)):
        print(f"\nStitching image {i+1} of {len(images)}")

        mkpts0, mkpts1 = deep_match_loftr(panorama, images[i])
        print("Matches found:", len(mkpts0))

        if len(mkpts0) < 20:
            print("Not enough matches, skipping image")
            continue

        H, mask = compute_homography_from_matches(mkpts0, mkpts1)
        if H is None:
            print("Homography failed")
            continue

        panorama = stitch_two_images(panorama, images[i], H)
        panorama = crop_black_borders(panorama)

        del mkpts0, mkpts1, H, mask
        gc.collect()

    return panorama


# 5. RUN

images = load_images(IMAGE_FOLDER, max_images=4)

if len(images) < 2:
    raise ValueError("Need at least 2 images")

show_images(images, titles=[f"Input {i+1}" for i in range(len(images))])

deep_panorama = deep_stitch_all(images)
deep_panorama = crop_black_borders(deep_panorama)

plt.figure(figsize=(16, 8))
plt.imshow(cv2.cvtColor(deep_panorama, cv2.COLOR_BGR2RGB))
plt.title("Deep Stitching Panorama")
plt.axis("off")
plt.show()

save_image(deep_panorama, "deep_panorama_output.jpg")
print("Done")

Loaded: 1.jpg | shape: (360, 640, 3)
Loaded: 2.jpg | shape: (360, 640, 3)
Total images loaded: 2
